In [40]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
from typing import Dict, List, Tuple
import kagglehub
from collections import defaultdict
import matplotlib.pyplot as plt
import networkx as nx

In [2]:
import warnings
warnings.filterwarnings("ignore")

# FiLMTest Original File

In [9]:
torch.set_printoptions(precision=4, sci_mode=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_PATH = "data/dolma_300_2024_1.2M.100_combined.txt"  # or GloVe path
EMB_DIM = 300

In [3]:
# Dataset downloading
path = kagglehub.dataset_download("nguyendatphik18ct/sdfghjjjjjj")
print(f"Dataset has been downloaded in: {path}")

file_path = os.path.join(path, "dolma_300_2024_1.2M.100_combined.txt")

Dataset has been downloaded in: /Users/mariia/.cache/kagglehub/datasets/nguyendatphik18ct/sdfghjjjjjj/versions/1


In [4]:
def load_embeddings(path: str) -> Dict[str, np.ndarray]:
    """
    Load word embeddings from a plain text file.

    Expected format:
        token dim1 dim2 ... dimD
    """
    embs: Dict[str, np.ndarray] = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 10:
                continue
            word = parts[0]
            vec = np.asarray(parts[1:], dtype=np.float32)
            embs[word] = vec
    return embs


def get_vec(word: str, embs: Dict[str, np.ndarray]) -> torch.Tensor:
    """Return embedding tensor for a given word."""
    if word not in embs:
        raise ValueError(f"Word '{word}' not found in embeddings.")
    return torch.tensor(embs[word], dtype=torch.float32, device=DEVICE)


print("Loading embeddings...")
embs = load_embeddings(file_path)
print("Loaded", len(embs), "tokens.")
EMB_DIM = len(next(iter(embs.values())))
print("Embedding dim:", EMB_DIM)


Loading embeddings...
Loaded 1200001 tokens.
Embedding dim: 300


In [5]:
OBJ_WORDS = [
    "cat", "dog", "animal", "pet", "mammal",
    "tiger", "lion", "wolf", "fox",
     "apple", "banana", "fruit", "produce",
    "car", "bus", "vehicle", "transport",
    "piano", "guitar", "instrument", "music",
    "red", "blue", "color",
    "happy", "sad", "emotion",
]

obj_words_valid = [w for w in OBJ_WORDS if w in embs]
print(f"obj_words_valid: {len(obj_words_valid)} words")

obj_words_valid: 27 words


In [6]:
TRIPLETS: List[Tuple[str, str, str]] = [
    ("cat", "dog", "animal"),
    ("tiger", "lion", "animal"),
    ("apple", "banana", "fruit"),
    ("car", "bus", "vehicle"),
    ("piano", "guitar", "instrument"),
    ("happy", "sad", "emotion"),
    ("red", "blue", "color"),
    ("king", "queen", "royalty"),
    ("seoul", "tokyo", "city"),
]


valid_triplets: List[Tuple[str, str, str]] = []
missing = set()

for w1, w2, wp in TRIPLETS:
    if w1 not in embs or w2 not in embs or wp not in embs:
        missing.update([w for w in [w1, w2, wp] if w not in embs])
        continue
    valid_triplets.append((w1, w2, wp))

print("Valid triplets:", len(valid_triplets))
if missing:
    print("Missing tokens (skipped):", missing)


Valid triplets: 9


In [7]:
# Cell 3: FiLM module (pure parent attraction, no repulsion)

class FiLMGen(nn.Module):
    """
    FiLM generator that maps (w1, w2) to a modulated vector z.

    z = gamma([w1, w2]) * base + beta([w1, w2]),
    where base = 0.5 * (w1 + w2).
    """
    def __init__(self, dim: int):
        super().__init__()
        hidden = 512
        self.gammanet = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim),
            nn.Sigmoid(),  # gating in [0,1]
        )
        self.betanet = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Linear(hidden, dim),
            nn.Tanh(),     # shift in [-1,1]
        )

    def forward(self, w1: torch.Tensor, w2: torch.Tensor) -> torch.Tensor:
        """
        w1, w2: (..., D)
        returns z: (..., D)
        """
        context = torch.cat([w1, w2], dim=-1)
        gamma = self.gammanet(context)
        beta = self.betanet(context)
        base = 0.5 * (w1 + w2)
        z = gamma * base + beta
        return z


def cosine(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Cosine similarity along last dim."""
    return F.cosine_similarity(a, b, dim=-1)


In [10]:
# Cell 4: Build training tensors from triplets

def build_training_tensors(
    triplets: List[Tuple[str, str, str]],
    embs: Dict[str, np.ndarray],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build (w1, w2, wp) tensors from word-based triplets.
    """
    v1_list, v2_list, vp_list = [], [], []
    for w1, w2, wp in triplets:
        v1_list.append(embs[w1])
        v2_list.append(embs[w2])
        vp_list.append(embs[wp])
    v1 = torch.tensor(np.stack(v1_list), dtype=torch.float32, device=DEVICE)
    v2 = torch.tensor(np.stack(v2_list), dtype=torch.float32, device=DEVICE)
    vp = torch.tensor(np.stack(vp_list), dtype=torch.float32, device=DEVICE)
    return v1, v2, vp


train_v1, train_v2, train_vp = build_training_tensors(valid_triplets, embs)
print("Train tensors shape:", train_v1.shape, train_v2.shape, train_vp.shape)


Train tensors shape: torch.Size([9, 300]) torch.Size([9, 300]) torch.Size([9, 300])


In [11]:
# Cell 5: Parent-attraction loss

class ParentAttractionLoss(nn.Module):
    """
    Simple cosine-based attraction loss:
        loss = 1 - cos(pred, target)
    """
    def __init__(self):
        super().__init__()

    def forward(
        self,
        pred: torch.Tensor,
        target: torch.Tensor,
    ) -> torch.Tensor:
        """
        pred, target: (B, D)
        """
        cos_sim = cosine(pred, target)  # (B,)
        loss = 1.0 - cos_sim
        return loss.mean()


In [13]:
# Cell 6: Train FiLM on hand-crafted triplets

model = FiLMGen(EMB_DIM).to(DEVICE)
criterion = ParentAttractionLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 2000

print("Training FiLM (pure parent attraction)...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    pred = model(train_v1, train_v2)      # (B, D)
    loss = criterion(pred, train_vp)

    loss.backward()
    optimizer.step()

    if epoch % 200 == 0 or epoch == 1:
        with torch.no_grad():
            cos_before = cosine(0.5 * (train_v1 + train_v2), train_vp).mean().item()
            cos_after = cosine(pred, train_vp).mean().item()
        print(
            f"[Epoch {epoch:4d}] loss = {loss.item():.4f} | "
            f"avg cos(avg,parent) = {cos_before:.4f} | "
            f"avg cos(FiLM,parent) = {cos_after:.4f}"
        )


Training FiLM (pure parent attraction)...
[Epoch    1] loss = 0.5966 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 0.4034
[Epoch  200] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  400] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  600] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch  800] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1000] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1200] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1400] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1600] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 1800] loss = 0.0000 | avg cos(avg,parent) = 0.7532 | avg cos(FiLM,parent) = 1.0000
[Epoch 2000] loss = 0.0000 | avg cos(avg,parent) = 0.7532 

In [14]:
v1 = get_vec("cat", embs)
v2 = get_vec("dog", embs)

with torch.no_grad():
    film_vec = model(v1.unsqueeze(0), v2.unsqueeze(0)).squeeze(0)

X_obj = torch.stack([get_vec(w, embs) for w in obj_words_valid])
sims_to_film = F.cosine_similarity(X_obj, film_vec.unsqueeze(0).expand(len(obj_words_valid), -1))

seed_film_cd = sims_to_film - sims_to_film.min()
seed_film_cd = seed_film_cd / (seed_film_cd.max() + 1e-8)

print(f"seed_film_cd посчитан! shape: {seed_film_cd.shape}")

seed_film_cd посчитан! shape: torch.Size([27])


In [15]:
# Cell 7: Top-k neighbor search

# Pre-build vocab tensor for neighbor lookup
all_words = list(embs.keys())
all_vecs = torch.tensor(
    np.stack([embs[w] for w in all_words], axis=0),
    dtype=torch.float32,
    device=DEVICE,
)
all_norm = F.normalize(all_vecs, p=2, dim=1)  # (V, D)


def topk_neighbors(
    query: torch.Tensor,
    k: int = 10,
) -> List[Tuple[str, float]]:
    """
    Return top-k neighbors and cosine scores for a query vector.

    query: (D,) or (1, D)
    """
    if query.dim() == 1:
        q = query.unsqueeze(0)
    else:
        q = query
    q_norm = F.normalize(q, p=2, dim=1)       # (1, D)
    sims = torch.matmul(all_norm, q_norm.T).squeeze(1)  # (V,)
    vals, idxs = torch.topk(sims, k=k)
    results: List[Tuple[str, float]] = []
    for v, idx in zip(vals.tolist(), idxs.tolist()):
        results.append((all_words[idx], float(v)))
    return results


In [16]:
# Cell 8: Evaluation — avg vs FiLM output

def evaluate_triplet(
    w1: str,
    w2: str,
    wp: str,
    model: FiLMGen,
    k: int = 10,
) -> None:
    """
    Print cosine to parent and top-k neighbors for
    - average vector
    - FiLM output
    """
    v1 = get_vec(w1, embs).unsqueeze(0)
    v2 = get_vec(w2, embs).unsqueeze(0)
    vp = get_vec(wp, embs).unsqueeze(0)

    v_avg = 0.5 * (v1 + v2)
    with torch.no_grad():
        z = model(v1, v2)

    cos_avg_parent = cosine(v_avg, vp).item()
    cos_z_parent = cosine(z, vp).item()
    cos_avg_child1 = cosine(v_avg, v1).item()
    cos_avg_child2 = cosine(v_avg, v2).item()
    cos_z_child1 = cosine(z, v1).item()
    cos_z_child2 = cosine(z, v2).item()

    print("=" * 60)
    print(f"Triplet: ({w1}, {w2}) -> {wp}")
    print(f"cos(avg, parent)  = {cos_avg_parent:.4f}")
    print(f"cos(FiLM, parent) = {cos_z_parent:.4f}")
    print(f"cos(avg, child1)  = {cos_avg_child1:.4f}")
    print(f"cos(avg, child2)  = {cos_avg_child2:.4f}")
    print(f"cos(FiLM, child1) = {cos_z_child1:.4f}")
    print(f"cos(FiLM, child2) = {cos_z_child2:.4f}")

    print("\nTop-k neighbors (avg):")
    n_avg = topk_neighbors(v_avg.squeeze(0), k=k)
    for i, (w, s) in enumerate(n_avg):
        print(f"  [{i:2d}] {w:15s}  {s:.4f}")

    print("\nTop-k neighbors (FiLM):")
    n_z = topk_neighbors(z.squeeze(0), k=k)
    for i, (w, s) in enumerate(n_z):
        print(f"  [{i:2d}] {w:15s}  {s:.4f}")


print("=== Evaluation on valid triplets ===")
for w1, w2, wp in valid_triplets:
    evaluate_triplet(w1, w2, wp, model, k=10)


=== Evaluation on valid triplets ===
Triplet: (cat, dog) -> animal
cos(avg, parent)  = 0.8186
cos(FiLM, parent) = 1.0000
cos(avg, child1)  = 0.9694
cos(avg, child2)  = 0.9748
cos(FiLM, child1) = 0.7894
cos(FiLM, child2) = 0.8013

Top-k neighbors (avg):
  [ 0] dog              0.9748
  [ 1] cat              0.9694
  [ 2] dogs             0.9095
  [ 3] cats             0.9014
  [ 4] pet              0.8743
  [ 5] puppy            0.8657
  [ 6] kitten           0.8384
  [ 7] pup              0.8340
  [ 8] pets             0.8322
  [ 9] kitty            0.8256

Top-k neighbors (FiLM):
  [ 0] animal           1.0000
  [ 1] animals          0.9289
  [ 2] pet              0.8200
  [ 3] human            0.8030
  [ 4] dog              0.8013
  [ 5] pets             0.7973
  [ 6] cats             0.7913
  [ 7] cat              0.7894
  [ 8] dogs             0.7887
  [ 9] humans           0.7827
Triplet: (tiger, lion) -> animal
cos(avg, parent)  = 0.6950
cos(FiLM, parent) = 1.0000
cos(avg, child1

In [18]:
def embed_query_to_extent(
    query_vec: torch.Tensor,
    X: torch.Tensor,
    temperature: float = 0.1,
) -> torch.Tensor:
    """
    Convert a query vector into a soft fuzzy extent over object pool X.

    query_vec : (D,)
    X         : (B, D)
    returns   : extent (B,) in [0, 1]
    """
    if query_vec.dim() == 1:
        q = query_vec.unsqueeze(0)
    else:
        q = query_vec
    B = X.shape[0]
    sims = cosine(
        X,
        q.expand(B, -1),
    )  # (B,)
    extent = F.softmax(sims / temperature, dim=0)
    return extent


# Readme Tasks

## 1. Unseen Child-Pair Generalization

In [19]:
UNSEEN_TEST = [
    # animal
    ("wolf", "fox", "animal"),
    ("cat", "tiger", "animal"),
    ("dog", "wolf", "animal"),
    ("rabbit", "hamster", "animal"),
    
    # fruit
    ("mango", "grape", "fruit"),
    ("pear", "orange", "fruit"),
    ("strawberry", "blueberry", "fruit"),
    
    # vehicle
    ("truck", "bus", "vehicle"),
    ("car", "motorcycle", "vehicle"),
    ("train", "subway", "vehicle"),
    
    # instrument
    ("violin", "cello", "instrument"),
    ("drums", "guitar", "instrument"),
    ("flute", "trumpet", "instrument"),
]

# Filtering non-present words
unseen_valid = []
for w1, w2, wp in UNSEEN_TEST:
    if w1 in embs and w2 in embs and wp in embs:
        unseen_valid.append((w1, w2, wp))
    else:
        missing = [w for w in [w1, w2, wp] if w not in embs]
        print(f"Skipping ({w1}, {w2}, {wp}) — missing: {missing}")

print(f"\nValid unseen pairs: {len(unseen_valid)}")

# Evaluation on unseen pairs

print("==================UNSEEN PAIR GENERALIZATION==================")


results_unseen = []

for w1, w2, wp in unseen_valid:
    v1 = get_vec(w1, embs).unsqueeze(0)
    v2 = get_vec(w2, embs).unsqueeze(0)
    vp = get_vec(wp, embs).unsqueeze(0)
    
    v_avg = 0.5 * (v1 + v2)
    with torch.no_grad():
        z = model(v1, v2)
    
    cos_avg = cosine(v_avg, vp).item()
    cos_film = cosine(z, vp).item()
    
    results_unseen.append((w1, w2, wp, cos_avg, cos_film))
    
    print(f"\n({w1}, {w2}) → {wp}")
    print(f"  cos(avg, parent)   = {cos_avg:.4f}")
    print(f"  cos(FiLM, parent)  = {cos_film:.4f}")
    print(f"  improvement        = {cos_film - cos_avg:+.4f}")

# Averaging results
avg_improvement = sum(r[4] - r[3] for r in results_unseen) / len(results_unseen)
print("\n" + "="*70)
print(f"Average improvement (FiLM over avg): {avg_improvement:+.4f}")
print(f"Success rate (cos(FiLM) > cos(avg)): {sum(r[4] > r[3] for r in results_unseen)}/{len(results_unseen)}")


Valid unseen pairs: 13
==================UNSEEN PAIR GENERALIZATION==================

(wolf, fox) → animal
  cos(avg, parent)   = 0.7112
  cos(FiLM, parent)  = 0.9193
  improvement        = +0.2081

(cat, tiger) → animal
  cos(avg, parent)   = 0.7784
  cos(FiLM, parent)  = 0.9827
  improvement        = +0.2042

(dog, wolf) → animal
  cos(avg, parent)   = 0.8176
  cos(FiLM, parent)  = 0.9723
  improvement        = +0.1547

(rabbit, hamster) → animal
  cos(avg, parent)   = 0.7343
  cos(FiLM, parent)  = 0.9438
  improvement        = +0.2096

(mango, grape) → fruit
  cos(avg, parent)   = 0.8767
  cos(FiLM, parent)  = 0.9444
  improvement        = +0.0677

(pear, orange) → fruit
  cos(avg, parent)   = 0.7874
  cos(FiLM, parent)  = 0.8558
  improvement        = +0.0683

(strawberry, blueberry) → fruit
  cos(avg, parent)   = 0.7916
  cos(FiLM, parent)  = 0.9331
  improvement        = +0.1415

(truck, bus) → vehicle
  cos(avg, parent)   = 0.8395
  cos(FiLM, parent)  = 0.9839
  improvement   

## Towards calculation of delta-stability

In [26]:
# 1. Defining attributes and their anchor words
ANCHORS = {
    "animalness": ["animal", "pet", "mammal"],
    "fruitness": ["fruit", "produce", "food"],
    "vehicleness": ["vehicle", "transport", "car"],
    "instrumentness": ["instrument", "music", "sound"],
    "colornes": ["color", "colour", "hue"],
    "emotionness": ["emotion", "feeling", "mood"],
}

# 2. Objects pool
OBJ_WORDS = [
    "cat", "dog", "animal", "pet", "mammal",
    "tiger", "lion", "wolf", "fox",
    "apple", "banana", "fruit", "produce",
    "car", "bus", "vehicle", "transport",
    "piano", "guitar", "instrument", "music",
    "red", "blue", "color",
    "happy", "sad", "emotion",
]

# 3. Filtering the objects
obj_words_valid = [w for w in OBJ_WORDS if w in embs]
print(f"FCA object pool: {len(obj_words_valid)} words")

# 4. Buiding the context
M = len(ANCHORS)
context = torch.zeros(len(obj_words_valid), M)
attr_names = list(ANCHORS.keys())

for j, (attr_name, anchor_words) in enumerate(ANCHORS.items()):
    valid_anchors = [a for a in anchor_words if a in embs]
    if not valid_anchors:
        continue
    anchor_vecs = torch.stack([get_vec(a, embs) for a in valid_anchors])
    anchor_center = anchor_vecs.mean(dim=0)
    X_obj = torch.stack([get_vec(w, embs) for w in obj_words_valid])
    sims = F.cosine_similarity(X_obj, anchor_center.unsqueeze(0).expand(len(obj_words_valid), -1))
    context[:, j] = sims

# 5. Normalizing in [0, 1]
context = (context - context.min()) / (context.max() - context.min() + 1e-8)
print(f"Context shape: {context.shape}")

FCA object pool: 27 words
Context shape: torch.Size([27, 6])


In [35]:
def get_seed_extent(w1, w2, model, obj_words, embs, device):
    """Creates seed_extent for pair of words through FiLM."""
    # word vectors
    v1 = torch.tensor(embs[w1], dtype=torch.float32, device=device).unsqueeze(0)
    v2 = torch.tensor(embs[w2], dtype=torch.float32, device=device).unsqueeze(0)
    
    # FiLM vector
    with torch.no_grad():
        film_vec = model(v1, v2).squeeze(0)
    
    # Cosine similarity with all objects
    X_obj = torch.stack([torch.tensor(embs[w], dtype=torch.float32, device=device) 
                         for w in obj_words])
    sims = F.cosine_similarity(X_obj, film_vec.unsqueeze(0).expand(len(obj_words), -1))
    
    # Normalization to [0,1]
    seed = (sims - sims.min()) / (sims.max() - sims.min() + 1e-8)
    return seed

In [52]:
def compute_all_concepts_with_delta(context, obj_names, attr_names, alphas=[0.6, 0.7, 0.8]):
    """
    Calculating all formal concepts and their  Δ-stability for different α.
    
    Returns:
        dict: {alpha: list of concepts with delta stability}
    """
    results = {}
    
    for alpha in alphas:
        print(f"\n{'='*70}")
        print(f"Analysis for α = {alpha}")
        print(f"{'='*70}")
        
        # Concept binarization
        binary_context = (context >= alpha).float()
        B, M = binary_context.shape
        
        # Finding all formal concepts
        concepts = find_all_formal_concepts(binary_context, obj_names, attr_names)
        
        print(f"Formal concepts found: {len(concepts)}")
        
        # For each concept calculating Δ-stability
        for concept in concepts:
            # Transforming extent and intent into tensors
            extent_mask = torch.zeros(B, dtype=torch.bool)
            for idx in concept['extent_indices']:
                extent_mask[idx] = True
            
            intent_mask = torch.zeros(M, dtype=torch.bool)
            for idx in concept['intent_indices']:
                intent_mask[idx] = True
            
            # Calculating  Δ-stability
            delta_info = compute_delta_stability_for_concept(
                extent_mask, intent_mask, binary_context, attr_names, obj_names
            )
            concept['delta_stability'] = delta_info
        
        results[alpha] = concepts
    
    return results


#================================================================================================

def find_all_formal_concepts(binary_context, obj_names, attr_names):
    """
    Finding all formal concepts in binary context
    Using Close-by-One (CbO) algorithm.
    """
    B, M = binary_context.shape
    binary_ctx = binary_context.byte()
    
    all_concepts = []

    def compute_intent(extent_indices):
        """Calculate intent by extent"""
        if len(extent_indices) == 0:
            return set(range(M))
        
        # Atributes that all objects in the extent posess
        intent = set(range(M))
        for obj_idx in extent_indices:
            obj_attrs = set(j for j in range(M) if binary_ctx[obj_idx, j] == 1)
            intent &= obj_attrs
        return intent
    
    def compute_extent(intent_indices):
        """Calculating extent by intent"""
        if len(intent_indices) == 0:
            return set(range(B))
        
        # Objects posessing all attributes from intent
        extent = set()
        for obj_idx in range(B):
            has_all = all(binary_ctx[obj_idx, j] == 1 for j in intent_indices)
            if has_all:
                extent.add(obj_idx)
        return extent
    
    def closure(intent_indices):
        """Closure of attribute set"""
        extent = compute_extent(intent_indices)
        return compute_intent(list(extent))
    
    def generate(prefix, candidates):
        """Recursive concept generation"""
        # Calculation prefix closure
        closed_intent = closure(prefix)
        closed_intent_list = sorted(closed_intent)
        
        # Adding concept if it doesn't exist yet
        extent = compute_extent(closed_intent_list)
        extent_list = sorted(extent)
        
        concept_key = (tuple(extent_list), tuple(closed_intent_list))
        
        if concept_key not in all_concepts:
            all_concepts.append(concept_key)
        
        # Iteration through the candidates
        for i, cand in enumerate(candidates):
            new_prefix = prefix + [cand]
            new_candidates = candidates[i+1:]
            
            # Canonicity check
            new_closed = closure(new_prefix)
            if len(new_closed) == len(closure(prefix + [cand])):
                generate(new_prefix, new_candidates)
    
    # Generation run
    generate([], list(range(M)))
    
    # Transform into comfortable format
    concepts = []
    for extent_indices, intent_indices in all_concepts:
        concepts.append({
            'extent': [obj_names[i] for i in extent_indices],
            'extent_indices': list(extent_indices),
            'intent': [attr_names[j] for j in intent_indices],
            'intent_indices': list(intent_indices),
            'extent_size': len(extent_indices),
            'intent_size': len(intent_indices)
        })
    
    return concepts
#================================================================================================

def compute_delta_stability_for_concept(extent_mask, intent_mask, binary_context, attr_names, obj_names):
    """
    Calculation  Δ-stability for one concept.
    """
    A_size = extent_mask.sum().item()
    
    # Attributes that don't exist in intent yet
    candidate_attrs = [j for j, is_in_intent in enumerate(intent_mask) if not is_in_intent]
    
    if not candidate_attrs:
        return {
            'delta': 0.0,
            'best_attr': None,
            'has_candidates': False,
            'candidates': [],
            'parent_objects': [obj_names[i] for i, v in enumerate(extent_mask) if v],
            'parent_count': A_size
        }
    
    candidates_info = []
    max_support = 0
    best_j = None
    
    for j in candidate_attrs:
        # Objects which have this attribute
        has_attr = (binary_context[:, j] == 1)
        
        # Objects that will stay after adding the attribute
        remaining = extent_mask & has_attr
        remaining_count = remaining.sum().item()
        remaining_indices = [i for i, v in enumerate(remaining) if v]
        remaining_names = [obj_names[i] for i in remaining_indices]
        
        # Objects lost
        lost = extent_mask & ~remaining
        lost_count = lost.sum().item()
        lost_indices = [i for i, v in enumerate(lost) if v]
        lost_names = [obj_names[i] for i in lost_indices]
        
        candidates_info.append({
            'attr_name': attr_names[j],
            'attr_index': j,
            'remaining_count': remaining_count,
            'remaining_names': remaining_names,
            'lost_count': lost_count,
            'lost_names': lost_names,
            'support': remaining_count,
            'would_delta': A_size - remaining_count
        })
        
        if remaining_count > max_support:
            max_support = remaining_count
            best_j = j
    
    delta = A_size - max_support
    best_attr = attr_names[best_j] if best_j is not None else None
    best_candidate = next((c for c in candidates_info if c['attr_index'] == best_j), None) if best_j else None
    
    return {
        'delta': delta,
        'best_attr': best_attr,
        'best_candidate': best_candidate,
        'has_candidates': True,
        'candidates': candidates_info,
        'parent_objects': [obj_names[i] for i, v in enumerate(extent_mask) if v],
        'parent_count': A_size
    }

#================================================================================================
def print_all_concepts_with_delta(results, min_extent_size=1, max_display=10):
    """
    Printing all concepts with their Δ-stability.
    """
    for alpha, concepts in results.items():
        print(f"\n{'='*100}")
        print(f"Results for  α = {alpha}")
        print(f"{'='*100}")
        print(f" Total concepts: {len(concepts)}")
        
        # Sorting by extent size from big to small
        concepts_sorted = sorted(concepts, key=lambda x: -x['extent_size'])
        
        # Filtering by minimal extent size
        concepts_filtered = [c for c in concepts_sorted if c['extent_size'] >= min_extent_size]
        
        print(f"Concepts with |Extent| >= {min_extent_size}: {len(concepts_filtered)}")
        
        for i, concept in enumerate(concepts_filtered[:max_display]):
            print(f"\n{'─'*100}")
            print(f"Concept {i+1}: |Extent|={concept['extent_size']}, |Intent|={concept['intent_size']}")
            print(f"{'─'*100}")
            
            # Extent
            extent_str = ', '.join(concept['extent'])
            if len(concept['extent']) > 8:
                extent_str += f" ... and also {len(concept['extent']) - 8}"
            print(f"Extent: {{{extent_str}}}")
            
            # Intent
            intent_str = ', '.join(concept['intent'])
            print(f" Intent: {{{intent_str}}}")
            
            # Δ-stability
            delta_info = concept['delta_stability']
            print(f"\n Δ-stability: {delta_info['delta']}")
            
            if delta_info['has_candidates'] and delta_info['best_attr']:
                print(f"   Best candidate for adding: '{delta_info['best_attr']}'")
                if delta_info['best_candidate']:
                    bc = delta_info['best_candidate']
                    print(f"     → Stay: {bc['remaining_count']} objects")
                    print(f"     → Lost: {bc['lost_count']} objects")
                    if bc['lost_names']:
                        print(f"     → Lost objects: {', '.join(bc['lost_names'])}")
            else:
                print(f"   → NO candidates for adding (the most specific concept")
        
        if len(concepts_filtered) > max_display:
            print(f"\n... and also {len(concepts_filtered) - max_display} concepts")
        
        # Statistics by  Δ-stability
        print(f"\n📈 Statistics Δ-stability:")
        deltas = [c['delta_stability']['delta'] for c in concepts_filtered]
        if deltas:
            print(f"   Average Δ: {sum(deltas)/len(deltas):.2f}")
            print(f"   Min Δ: {min(deltas):.2f}")
            print(f"   Max Δ: {max(deltas):.2f}")
            print(f"   Stable (Δ≤2): {sum(1 for d in deltas if d <= 2)}")
            print(f"   Unstable (Δ>5): {sum(1 for d in deltas if d > 5)}")

In [53]:
def analyze_with_seed(seed_extent, context, obj_names, attr_names, alphas=[0.6, 0.7, 0.8]):
    """
    Analysing all concepts and finding those which closer to seed_extent.
    """
    results = {}
    
    for alpha in alphas:
        print(f"\n{'='*100}")
        print(f"α = {alpha}")
        print(f"{'='*100}")
        
        binary_context = (context >= alpha).float()
        
        # Finding all concepts
        all_concepts = find_all_formal_concepts(binary_context, obj_names, attr_names)
        
        # Defining seed_indices
        seed_indices = set((seed_extent > 0.5).nonzero().squeeze().tolist())
        
        # For all concepts calculate its intersection with  seed
        for concept in all_concepts:
            extent_set = set(concept['extent_indices'])
            
            # Jaccard similarity с seed
            intersection = len(seed_indices & extent_set)
            union = len(seed_indices | extent_set)
            jaccard = intersection / union if union > 0 else 0
            
            # Adding metrics
            concept['seed_overlap'] = intersection
            concept['seed_jaccard'] = jaccard
            
            # Calculating Δ-stability
            extent_mask = torch.zeros(len(obj_names), dtype=torch.bool)
            for idx in concept['extent_indices']:
                extent_mask[idx] = True
            
            intent_mask = torch.zeros(len(attr_names), dtype=torch.bool)
            for idx in concept['intent_indices']:
                intent_mask[idx] = True
            
            delta_info = compute_delta_stability_for_concept(
                extent_mask, intent_mask, binary_context, attr_names, obj_names
            )
            concept['delta_stability'] = delta_info
        
        # Sorting by Jaccard similarity
        all_concepts.sort(key=lambda x: -x['seed_jaccard'])
        
        results[alpha] = all_concepts
        
        # Printng top-5 closest to seed concepts
        print(f"\ top-5 concepts, closest to SEED:")
        print(f"{'─'*60}")
        
        for i, concept in enumerate(all_concepts[:5]):
            print(f"\n{i+1}. Jaccard = {concept['seed_jaccard']:.3f} (intersection: {concept['seed_overlap']} objects)")
            print(f"   Extent ({concept['extent_size']}): {concept['extent'][:8]}")
            print(f"   Intent: {concept['intent']}")
            print(f"   Δ-stability: {concept['delta_stability']['delta']}")
            if concept['delta_stability']['best_attr']:
                print(f"   Unstable attribute: {concept['delta_stability']['best_attr']}")
    
    return results
    

In [54]:
def compare_concepts_stability(concepts_by_alpha):
    """
    Comparing concept stability for different α.
    """
    print("\n" + "="*100)
    print("Comparing concept stability for different α.")
    print("="*160)
    
    for alpha, concepts in concepts_by_alpha.items():
        # Grouping concepts by intent
        intent_groups = {}
        for concept in concepts:
            intent_key = tuple(sorted(concept['intent']))
            if intent_key not in intent_groups:
                intent_groups[intent_key] = []
            intent_groups[intent_key].append(concept)
        
        print(f"\n{'─'*100}")
        print(f"α = {alpha}")
        print(f"{'─'*100}")
        
        # For each intent show concepts
        for intent_key, concept_list in sorted(intent_groups.items(), key=lambda x: -len(x[1][0]['extent'])):
            if not intent_key:
                continue
            
            print(f"\nIntent: {list(intent_key)}")
            for concept in concept_list:
                print(f"  |Extent|={concept['extent_size']}, Δ={concept['delta_stability']['delta']:.1f}")
                print(f"   Objects: {concept['extent']}")


# Analysis run

print("Analysis run")



results_all = compute_all_concepts_with_delta(
    context, obj_words_valid, attr_names, 
    alphas=[0.5, 0.6, 0.7, 0.8]
)

# Results
print_all_concepts_with_delta(results_all, min_extent_size=1, max_display=15)

# Analysis with seed
seed_film_cd = get_seed_extent("cat", "dog", model, obj_words_valid, embs, DEVICE)

results_with_seed = analyze_with_seed(
    seed_film_cd, context, obj_words_valid, attr_names,
    alphas=[0.5, 0.6, 0.7, 0.8]
)

# Stability comparison
compare_concepts_stability(results_with_seed)

Analysis run

Analysis for α = 0.5
Formal concepts found: 17

Analysis for α = 0.6
Formal concepts found: 12

Analysis for α = 0.7
Formal concepts found: 8

Analysis for α = 0.8
Formal concepts found: 8

Results for  α = 0.5
 Total concepts: 17
Concepts with |Extent| >= 1: 17

────────────────────────────────────────────────────────────────────────────────────────────────────
Concept 1: |Extent|=27, |Intent|=0
────────────────────────────────────────────────────────────────────────────────────────────────────
Extent: {cat, dog, animal, pet, mammal, tiger, lion, wolf, fox, apple, banana, fruit, produce, car, bus, vehicle, transport, piano, guitar, instrument, music, red, blue, color, happy, sad, emotion ... and also 19}
 Intent: {}

 Δ-stability: 14
   Best candidate for adding: 'fruitness'
     → Stay: 13 objects
     → Lost: 14 objects
     → Lost objects: mammal, tiger, lion, wolf, fox, car, bus, vehicle, piano, guitar, instrument, music, sad, emotion

───────────────────────────────